In [ ]:
HOST = "m4-macbook-air"  # a folder under backend/benchmark-results/

In [ ]:
import warnings
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

from benchlib import load, plots, random_suite as rs, traffic_suite as ts
from benchlib.hosts import HOSTS, REFERENCE

warnings.filterwarnings("ignore", category=RuntimeWarning)
plots.style()
pd.set_option("display.precision", 1)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)


def show(fig):
    if fig is not None:
        display(fig)
        plt.close(fig)

d = load.load(HOST)
h = d.host
R, T = d.random, d.traffic

In [ ]:
lines = [
    f"# {h.name}: benchmark report",
    "",
    f"- **Machine:** {h.hardware}; {h.os}; cooling: {h.cooling}",
    f"- **Docker:** {h.docker}, resources **{h.resources}**",
]
for name, suite in (("Idle capacity (`random`)", R), ("Traffic limits (`traffic`)", T)):
    if suite is None:
        continue
    m = suite.manifest
    lines.append(
        f"- **{name}:** `{suite.dir.name}`, AE3GIS `{m['git']['commit'][:8]}`, "
        f"{str(m.get('started_at', ''))[:10]} → {str(m.get('ended_at') or '')[:10] or 'running'}"
    )
if T is not None:
    lines.append(f"- **Traffic cells final:** {int(T.cells['done'].sum())} of {len(T.cells)}")
if h.deviations:
    lines += ["", "**Settings that differ from the shared suites:**", ""] + [f"- {x}" for x in h.deviations]
if h.issues:
    lines += ["", "**Read these numbers with:**", ""] + [f"- {x}" for x in h.issues]
lines += ["", f"Notes from the run: `backend/benchmark-results/{h.key}/NOTES.md`."]
display(Markdown("\n".join(lines)))

In [ ]:
# Sanity checks: the data is complete and matches what the runner wrote.
import json
assert R is not None and len(R.climbs) == 10, "10 random climbs"
assert T is not None and len(T.cells) == 100, "100 traffic cells"
ref = T.dir / "limits-burst-limits.json"
if T.cells["done"].all() and ref.exists():
    want = {r["id"]: (r["comfortable"], r["fails_at"]) for r in json.loads(ref.read_text())}
    got = {c.cell: (None if pd.isna(c.comfortable) else int(c.comfortable),
                    None if pd.isna(c.fails_at) else int(c.fails_at)) for c in T.cells.itertuples()}
    assert got == {k: tuple(v) for k, v in want.items()}, "grids match limits-burst-limits.json"
print("checks passed:", len(R.climbs), "climbs,", len(T.cells), "traffic cells,",
      int(T.cells["done"].sum()), "final")

## 1. Idle capacity (`random` suite)

How many nodes this host holds when they only idle.
- **Census:** deploys each node type · image on its own, to see what one node costs and which images run here.
- **Climbs:** 10 climbs deploy a random mix of every type (equal odds, then an image of that type) at growing sizes, until a limit stops them (memory, a deploy that fails or takes too long, disk). The largest size that passed is the climb's **ceiling**.

Every climb counts in the statistics. Climbs with known problems are greyed and annotated.

### Census: what one node of each image costs

In [ ]:
c = rs.census_table(R)
display(c.style.format({"host_mb_per_node": "{:.1f}", "cgroup_mb_per_node": "{:.1f}", "deploy_s": "{:.1f}",
                        "ready_s": "{:.2f}", "disk_mb_per_node": "{:.1f}"}, na_rep="–").hide(axis="index"))
show(plots.census_bars(c, f"{h.name}: host memory per node, by image"))

The census deploys only 5 nodes per image, so for light images the host figure above is mostly the lab's fixed overhead. Each image's **container memory**, measured over every container at the climbs' ceilings (hundreds of nodes), is the sturdier per-image cost:

In [ ]:
im = rs.image_memory(R).sort_values(ascending=False).rename("container MB (median over climbs)")
display(im.to_frame().T.style.format("{:.1f}"))

### Climbs

In [ ]:
st = rs.ceiling_stats(R)
display(Markdown(f"**Ceiling over all {st['climbs']:.0f} climbs: {st['mean']:.0f} ± {st['std']:.0f} hosts** "
                 f"(range {st['min']:.0f}–{st['max']:.0f})."))
t = rs.climb_table(R).assign(stop=rs.climb_stops(R).values)
display(t[["seed", "ceiling", "nodes", "mem_pct", "mb_per_node", "deploy_s_per_node", "stop", "note"]]
        .style.format({"mem_pct": "{:.1f}", "mb_per_node": "{:.1f}", "deploy_s_per_node": "{:.2f}"}, na_rep="–"))
show(plots.ceilings(R.climbs, h.color, f"{h.name}: idle ceilings"))
show(plots.trajectories(R.steps, R.climbs, f"{h.name}: climbs step by step"))

### What drives the spread between climbs

Each climb draws a different mix. The scatter below sets its ceiling against what one node of that mix costs, from the census. The table lists the images whose share of the mix moves the ceiling most.

In [ ]:
cc = rs.composition_cost(R)
show(plots.ceiling_vs_mix(cc, h.color, f"{h.name}: ceiling vs the cost of the mix"))
display(rs.drivers(R).style.format({"corr_with_ceiling": "{:+.2f}", "host_mb_per_node": "{:.1f}"}).hide(axis="index"))

### Share of each type at the ceiling (%)

In [ ]:
display(rs.type_shares(R).T)

## 2. Traffic limits (`traffic` suite)

100 cells: 5 rates × 5 burst intervals × TCP/UDP × two patterns.
- **clients → servers:** one flow per host, north–south through the routers and core.
- **mesh:** two flows per host, east–west inside a LAN.

For each cell, the search finds the most hosts whose traffic still gets through (≥ 95% delivered, ≤ 1% UDP loss) and where it fails, 50 hosts apart; the comfortable size passes twice. Grid markers:
- `mem`: a host limit (memory, a deploy that failed or took too long), not traffic;
- `tool`: clients not started in time;
- `≥ N`: never failed up to N;
- `…`: still being searched.

In [ ]:
display(ts.overview(T).to_frame('').style)

### Limits: comfortable / fails at (hosts)

In [ ]:
show(plots.limit_heatmaps(T.cells, f"{h.name}: traffic limits"))

### Traffic carried at the comfortable size (Mb/s offered = hosts × rate × flows per host)

In [ ]:
grids = {s: ts.grid(T.cells, s, "carried_mbps") for s in ts.SCENARIO_ORDER}
show(plots.value_heatmaps(grids, f"{h.name}: traffic carried at the comfortable size", unit="Mb/s", cmap="cividis"))

### What the traffic costs

How much of the host's CPU the lab's containers use per 100 Mb/s delivered, by scenario and lab size (passing runs). On one machine it should be roughly flat.

A cost that falls with lab size is the signature of partial load: under the old driver, big labs' clients were still starting during the measured window. Nodes receive each delivered byte several times: once per hop.

In [ ]:
cost = ts.cost_by_size(T.rows)
display(cost.style.format("{:.2f}", na_rep="–"))
show(plots.cost_lines(cost, f"{h.name}: lab CPU per 100 Mb/s by lab size"))
k = ts.kept(T.rows)
display(k[k["outcome"] == "ok"].groupby("scenario")["rx_per_delivered"].median()
        .rename("node receives per delivered byte").to_frame().T.style.format("{:.1f}"))

### What set each cell's limit

In [ ]:
display(ts.reasons(T.cells))
f = ts.at_limit(T, "fails")
if not f.empty:
    display(f.groupby(["scenario", "reason"])[["cpu_pct", "mem_pct", "delivered_ratio", "loss_pct"]]
            .median().style.format({"cpu_pct": "{:.0f}", "mem_pct": "{:.0f}", "delivered_ratio": "{:.0%}",
                                    "loss_pct": "{:.1f}"}, na_rep="–")
            .set_caption("At the size each cell fails at (medians)"))

### Latency (TCP) and loss (UDP) at the comfortable size

In [ ]:
ok = ts.at_limit(T, "comfortable")
if not ok.empty:
    pivot = lambda s, v: ok[ok["scenario"] == s].pivot(index="rate", columns="interval_ms", values=v)
    show(plots.value_heatmaps({s: pivot(s, "rtt_p95_ms") for s in ("cs·tcp", "mesh·tcp")},
                              f"{h.name}: TCP RTT p95 at the comfortable size", unit="ms", log=True))
    show(plots.value_heatmaps({s: pivot(s, "loss_pct") for s in ("cs·udp", "mesh·udp")},
                              f"{h.name}: UDP loss at the comfortable size", fmt="{:.2f}", unit="%"))

### The burst interval's effect

In [ ]:
show(plots.interval_lines(T.cells, f"{h.name}: comfortable hosts across burst intervals"))

### What a lab costs to deploy, by size

In [ ]:
show(plots.deploy_costs(T.deploys, f"{h.name}: lab cost by size (median per size)"))

### How the search went (every counted probe per cell)

In [ ]:
for s in ts.SCENARIO_ORDER:
    show(plots.search_history(ts.search_history(T, s), f"{h.name}: {plots._scenario(s)}"))

## 3. Data quality

Which traffic driver measured each run, and the old driver's symptoms in mesh labs, by lab size:
- **dead flows:** at least 5% of flows delivered nothing;
- **memory stalls:** full PSI.

Under the fixed driver every client starts before the window, so dead flows there are a real overload: clients that couldn't connect.

In [ ]:
for pattern in ("mesh", "cs"):
    q = ts.quality_by_size(T.rows, pattern)
    if not q.empty:
        display(q.style.format({"dead_flows_pct": "{:.0f}", "psi_mem_full_median": "{:.1f}",
                                "mem_pct_median": "{:.0f}"}).set_caption(f"{pattern} labs"))
if len(T.dropped):
    display(T.dropped.groupby(["by", "why"]).size().rename("results dropped").to_frame())